# Instruction-Tuned SLM: SFT + DPO — Kaggle Training Notebook

Takes a raw pretrained model that generates fluent text but doesn't reliably follow
instructions, and turns it into a small instruction-tuned assistant in two stages:

1. **SFT** — teach it the *shape* of a good response (instruction in, helpful answer out)
2. **DPO** — teach it *which* of two responses a human would prefer

...then proves the improvement with numbers: **base vs. SFT vs. SFT+DPO**, scored by
an LLM judge plus an objective, judge-independent metric.

**Preference axis targeted: conciseness.** DPO is expected to reduce mean response
length / padding while holding or improving judge win rate — see the `PREFERENCE_AXIS`
config value below. Whatever the actual result turns out to be, write it up honestly in
the final cell — a null DPO result is a finding, not a failed project.

**Before running:**
1. Turn on the **GPU T4 x1** accelerator (Settings → Accelerator).
2. Add a Kaggle secret named `HF_TOKEN` (a Hugging Face *write* token) via
   Add-ons → Secrets, so this notebook can push LoRA adapters to the Hub after every
   stage — Kaggle sessions die, adapters are ~50MB, don't rely on `/kaggle/working` alone.
3. If you want an OpenAI judge instead of the default local HF judge, also add an
   `OPENAI_API_KEY` secret and flip `JUDGE_BACKEND` to `"openai"` in the config cell.
4. Set `HF_USERNAME` in the config cell to your Hugging Face username.

**Total GPU time if all cells run once:** ~3-4 hrs, comfortably inside Kaggle's 30
GPU-hr/week free-tier quota.


## 1. Environment setup (Day 1)

In [ ]:
!pip install -q -U "transformers==4.46.*" "trl==0.12.*" "peft==0.13.*" "datasets==3.0.*" "accelerate==1.0.*" "bitsandbytes==0.44.*" "huggingface_hub>=0.25" sentencepiece protobuf
# Versions pinned deliberately (see README) — TRL's DPOTrainer/SFTTrainer API has
# changed argument names across minor versions (DPOConfig/SFTConfig, tokenizer= ->
# processing_class=). Half the answers you'll find online are for a version you
# aren't running.


In [ ]:
import gc
import json
import os
import random
import re

import torch

print("torch:", torch.__version__)
assert torch.cuda.is_available(), "No GPU visible — enable the T4 x1 accelerator in notebook settings."
print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
# --- Hugging Face login (Day 1 / §2) -----------------------------------
# Uses a Kaggle secret so the token is never hardcoded in the notebook.
from huggingface_hub import login as hf_login

try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    hf_token = os.environ.get("HF_TOKEN")

if hf_token:
    hf_login(token=hf_token)
    print("Logged in to Hugging Face Hub.")
else:
    print("WARNING: no HF_TOKEN secret found — adapter pushes to the Hub will fail. "
          "Add one via Add-ons > Secrets if you want Hub checkpointing.")


In [ ]:
# --- Config (mirrors src/config.py in the repo) -------------------------

PREFERENCE_AXIS = "conciseness"  # decide this up front — see project breakdown §1

BASE_MODEL = "Qwen/Qwen2.5-1.5B"
JUDGE_BACKEND = "hf_local"          # "hf_local" or "openai"
JUDGE_MODEL_HF = "Qwen/Qwen2.5-7B-Instruct"
JUDGE_MODEL_OPENAI = "gpt-4o-mini"

SFT_DATASET = "tatsu-lab/alpaca"
DPO_DATASET = "Intel/orca_dpo_pairs"

SFT_TRAIN_SIZE = 2500
DPO_TRAIN_SIZE = 1500
EVAL_PROMPTS_SIZE = 40
PREF_HOLDOUT_SIZE = 200

SEED = 42
random.seed(SEED)

LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj"]

SFT_LEARNING_RATE = 2e-4
SFT_TRAIN_BATCH_SIZE = 4
SFT_GRAD_ACCUM_STEPS = 4
SFT_EPOCHS = 2
SFT_MAX_SEQ_LENGTH = 512
SFT_WARMUP_RATIO = 0.03
SFT_LR_SCHEDULER = "cosine"

DPO_BETA = 0.1
DPO_LEARNING_RATE = 5e-6
DPO_TRAIN_BATCH_SIZE = 2
DPO_GRAD_ACCUM_STEPS = 4
DPO_EPOCHS = 1
DPO_MAX_LENGTH = 1024
DPO_MAX_PROMPT_LENGTH = 512

GEN_MAX_NEW_TOKENS = 256
GEN_TEMPERATURE = 0.7
GEN_TOP_P = 0.9

HF_USERNAME = "your-hf-username"  # <-- set this
SFT_ADAPTER_REPO = f"{HF_USERNAME}/qwen2.5-1.5b-sft-alpaca-lora"
DPO_ADAPTER_REPO = f"{HF_USERNAME}/qwen2.5-1.5b-sft-dpo-orca-lora"

OUTPUT_DIR = "/kaggle/working"
SFT_ADAPTER_DIR = f"{OUTPUT_DIR}/sft_adapter"
DPO_ADAPTER_DIR = f"{OUTPUT_DIR}/dpo_adapter"
EVAL_DIR = f"{OUTPUT_DIR}/eval"
os.makedirs(EVAL_DIR, exist_ok=True)

print(f"Preference axis: {PREFERENCE_AXIS}")
print(f"Base model: {BASE_MODEL}")
print(f"Judge backend: {JUDGE_BACKEND}")


## 2. Data (Day 1)

Held-out slices are carved out **first**, before any subsampling of the training
data — so the eval prompts and the preference holdout can never leak into what the
model actually trains on.

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


def _format_instruction(example):
    if example.get("input"):
        return f"{example['instruction']}\n\n{example['input']}"
    return example["instruction"]


def load_sft_splits():
    raw = load_dataset(SFT_DATASET, split="train")
    raw = raw.filter(lambda ex: len(ex["output"].strip()) > 0)

    rng = random.Random(SEED)
    indices = list(range(len(raw)))
    rng.shuffle(indices)

    eval_indices = indices[:EVAL_PROMPTS_SIZE]
    remaining = indices[EVAL_PROMPTS_SIZE:]
    train_indices = remaining[:SFT_TRAIN_SIZE]

    eval_prompts = [_format_instruction(raw[i]) for i in eval_indices]
    train_raw = raw.select(train_indices)

    def to_text(example):
        messages = [
            {"role": "user", "content": _format_instruction(example)},
            {"role": "assistant", "content": example["output"]},
        ]
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        return {"text": text}

    train_dataset = train_raw.map(to_text, remove_columns=train_raw.column_names)
    return train_dataset, eval_prompts


def load_dpo_splits():
    raw = load_dataset(DPO_DATASET, split="train")

    def build(example):
        system = example.get("system") or ""
        question = example["question"]
        messages = []
        if system.strip():
            messages.append({"role": "system", "content": system})
        messages.append({"role": "user", "content": question})
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        return {"prompt": prompt, "chosen": example["chosen"], "rejected": example["rejected"]}

    formatted = raw.map(build, remove_columns=raw.column_names)

    rng = random.Random(SEED)
    indices = list(range(len(formatted)))
    rng.shuffle(indices)

    holdout_indices = indices[:PREF_HOLDOUT_SIZE]
    remaining = indices[PREF_HOLDOUT_SIZE:]
    train_indices = remaining[:DPO_TRAIN_SIZE]

    pref_holdout = [formatted[i] for i in holdout_indices]
    train_dataset = formatted.select(train_indices)
    return train_dataset, pref_holdout


sft_train_dataset, eval_prompts = load_sft_splits()
dpo_train_dataset, pref_holdout = load_dpo_splits()

print(f"SFT train examples: {len(sft_train_dataset)}")
print(f"DPO train pairs: {len(dpo_train_dataset)}")
print(f"Held-out eval prompts: {len(eval_prompts)}")
print(f"Preference holdout pairs: {len(pref_holdout)}")


In [ ]:
# Read these character by character: verify special tokens, EOS placement,
# and that the prompt/response boundary is where you think it is.
for i in range(5):
    print(f"===== example {i} =====")
    print(repr(sft_train_dataset[i]["text"]))
    print()


## 3. Eval harness (Day 2 — built BEFORE there's anything to judge)

This is the most important reordering in the whole plan: the judge pipeline gets
built and smoke-tested before any training happens, so a broken harness is a Day 2
problem, not a Day 6 one.

In [ ]:
# --- Model loading helpers (T4-safe: fp16, sdpa attention) --------------
from transformers import AutoModelForCausalLM
from peft import PeftModel


def load_base_model():
    return AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        torch_dtype=torch.float16,
        attn_implementation="sdpa",
        device_map="auto",
    )


def load_with_adapter(adapter_path):
    base = load_base_model()
    model = PeftModel.from_pretrained(base, adapter_path)
    model.eval()
    return model


def free_model(model):
    del model
    gc.collect()
    torch.cuda.empty_cache()


In [ ]:
# --- Generation helper ---------------------------------------------------

@torch.no_grad()
def generate_batch(model, prompts, max_new_tokens=GEN_MAX_NEW_TOKENS,
                    temperature=GEN_TEMPERATURE, top_p=GEN_TOP_P):
    """Returns list of {"prompt", "response", "hit_eos"}. hit_eos feeds the
    format-adherence metric (§6)."""
    model.eval()
    device = next(model.parameters()).device
    results = []

    for prompt in prompts:
        messages = [{"role": "user", "content": prompt}]
        templated = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(templated, return_tensors="pt").to(device)
        input_len = inputs["input_ids"].shape[1]

        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=top_p,
            pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
        )

        new_tokens = output_ids[0][input_len:]
        response = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
        hit_eos = bool(len(new_tokens) < max_new_tokens and tokenizer.eos_token_id in new_tokens.tolist())
        results.append({"prompt": prompt, "response": response, "hit_eos": hit_eos})

    return results


In [ ]:
# --- LLM judge -------------------------------------------------------------
# Blind, order-randomized per prompt (never always A-first — this is the
# position-bias fix). JSON parsing is defensive: judges return markdown-fenced
# JSON or trailing prose often enough that a bare json.loads() will kill a
# 40-prompt run partway through.

JUDGE_PROMPT_TEMPLATE = """You are comparing two responses to the same instruction. Decide which response
is more helpful, accurate, and appropriately concise. A longer response is not
automatically better — penalize padding, restatement, and unnecessary preamble.
Ignore which response is listed first; the order is randomized.

Instruction: {prompt}

Response A: {response_a}

Response B: {response_b}

Respond with JSON only, no markdown fences, no other text:
{{"winner": "A" | "B" | "tie", "reason": "one sentence"}}
"""


def _parse_judge_json(raw_text):
    text = raw_text.strip()
    text = re.sub(r"^```(?:json)?", "", text).strip()
    text = re.sub(r"```$", "", text).strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group(0))
        except json.JSONDecodeError:
            pass
    return None


_judge_model = None
_judge_tokenizer = None
_openai_client = None


def load_local_judge():
    global _judge_model, _judge_tokenizer
    from transformers import BitsAndBytesConfig

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type="nf4",
    )
    _judge_tokenizer = AutoTokenizer.from_pretrained(JUDGE_MODEL_HF)
    _judge_model = AutoModelForCausalLM.from_pretrained(
        JUDGE_MODEL_HF,
        quantization_config=bnb_config,
        device_map="auto",
        attn_implementation="sdpa",
    )
    _judge_model.eval()
    return _judge_model, _judge_tokenizer


def load_openai_judge():
    global _openai_client
    import openai

    try:
        from kaggle_secrets import UserSecretsClient
        api_key = UserSecretsClient().get_secret("OPENAI_API_KEY")
    except Exception:
        api_key = os.environ.get("OPENAI_API_KEY")
    _openai_client = openai.OpenAI(api_key=api_key)
    return _openai_client


def _call_local_judge(prompt_text):
    messages = [{"role": "user", "content": prompt_text}]
    templated = _judge_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = _judge_tokenizer(templated, return_tensors="pt").to(next(_judge_model.parameters()).device)
    with torch.no_grad():
        output_ids = _judge_model.generate(
            **inputs, max_new_tokens=128, do_sample=False,
            pad_token_id=_judge_tokenizer.pad_token_id or _judge_tokenizer.eos_token_id,
        )
    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    return _judge_tokenizer.decode(new_tokens, skip_special_tokens=True)


def _call_openai_judge(prompt_text):
    resp = _openai_client.chat.completions.create(
        model=JUDGE_MODEL_OPENAI,
        messages=[{"role": "user", "content": prompt_text}],
        temperature=0.0,
    )
    return resp.choices[0].message.content


def judge_pair(prompt, response_a, response_b, backend=None, rng=None):
    """Returns {"winner": "a"|"b"|"tie", "reason", "parsed_ok"} — "a"/"b" refer
    to the ORIGINAL arguments, not the randomized slots shown to the judge."""
    backend = backend or JUDGE_BACKEND
    rng = rng or random

    swapped = rng.random() < 0.5
    shown_a, shown_b = (response_b, response_a) if swapped else (response_a, response_b)
    prompt_text = JUDGE_PROMPT_TEMPLATE.format(prompt=prompt, response_a=shown_a, response_b=shown_b)

    parsed, raw = None, ""
    for _ in range(2):
        raw = _call_local_judge(prompt_text) if backend == "hf_local" else _call_openai_judge(prompt_text)
        parsed = _parse_judge_json(raw)
        if parsed is not None and "winner" in parsed:
            break

    if parsed is None:
        return {"winner": "tie", "reason": "unparseable judge output", "parsed_ok": False, "raw": raw}

    shown_winner = str(parsed.get("winner", "tie")).strip().upper()
    if shown_winner == "A":
        winner = "b" if swapped else "a"
    elif shown_winner == "B":
        winner = "a" if swapped else "b"
    else:
        winner = "tie"

    return {"winner": winner, "reason": parsed.get("reason", ""), "parsed_ok": True, "raw": raw}


def summarize_results(results, name_a="A", name_b="B"):
    wins_a = sum(1 for r in results if r["winner"] == "a")
    wins_b = sum(1 for r in results if r["winner"] == "b")
    ties = sum(1 for r in results if r["winner"] == "tie")
    decisive = wins_a + wins_b
    return {
        f"{name_a}_wins": wins_a,
        f"{name_b}_wins": wins_b,
        "ties": ties,
        "total": len(results),
        f"{name_a}_win_rate_excl_ties": (wins_a / decisive) if decisive else float("nan"),
    }


if JUDGE_BACKEND == "hf_local":
    load_local_judge()
elif JUDGE_BACKEND == "openai":
    load_openai_judge()
print("Judge backend ready:", JUDGE_BACKEND)


In [ ]:
# --- Day 2 smoke test: judge base vs. base, expect ~50/50 with ties -----
_smoke_base_model = load_base_model()
_smoke_gens_a = generate_batch(_smoke_base_model, eval_prompts)
_smoke_gens_b = generate_batch(_smoke_base_model, eval_prompts)
free_model(_smoke_base_model)

_smoke_results = [
    judge_pair(p, a["response"], b["response"])
    for p, a, b in zip(eval_prompts, _smoke_gens_a, _smoke_gens_b)
]
_smoke_summary = summarize_results(_smoke_results, name_a="sample1", name_b="sample2")
print(json.dumps(_smoke_summary, indent=2))
n_unparsed = sum(1 for r in _smoke_results if not r["parsed_ok"])
print(f"Unparseable judge responses (logged as ties): {n_unparsed}/{len(_smoke_results)}")
print("Expect roughly 50/50 with plenty of ties. 90/10 here means the harness is broken — fix before training.")


## 4. LoRA SFT (Day 3)

In [ ]:
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

sft_model = load_base_model()
sft_model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
sft_model.config.use_cache = False

peft_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGET_MODULES,
    bias="none",
    task_type="CAUSAL_LM",
)

sft_config = SFTConfig(
    output_dir=SFT_ADAPTER_DIR,
    per_device_train_batch_size=SFT_TRAIN_BATCH_SIZE,
    gradient_accumulation_steps=SFT_GRAD_ACCUM_STEPS,
    num_train_epochs=SFT_EPOCHS,
    learning_rate=SFT_LEARNING_RATE,
    fp16=True,  # T4 is Turing: no bf16 support
    max_seq_length=SFT_MAX_SEQ_LENGTH,
    warmup_ratio=SFT_WARMUP_RATIO,
    lr_scheduler_type=SFT_LR_SCHEDULER,
    dataset_text_field="text",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    logging_steps=10,
    save_strategy="epoch",
    report_to="none",
)

sft_trainer = SFTTrainer(
    model=sft_model,
    args=sft_config,
    train_dataset=sft_train_dataset,
    peft_config=peft_config,
    processing_class=tokenizer,
)

sft_trainer.train()


In [ ]:
sft_trainer.model.save_pretrained(SFT_ADAPTER_DIR)
tokenizer.save_pretrained(SFT_ADAPTER_DIR)
print(f"SFT adapter saved to {SFT_ADAPTER_DIR}")

if hf_token:
    sft_trainer.model.push_to_hub(SFT_ADAPTER_REPO)
    tokenizer.push_to_hub(SFT_ADAPTER_REPO)
    print(f"Pushed SFT adapter to https://huggingface.co/{SFT_ADAPTER_REPO}")
else:
    print("Skipping Hub push (no HF_TOKEN). Adapter is saved locally.")

free_model(sft_trainer.model)
del sft_trainer
gc.collect()
torch.cuda.empty_cache()


## 5. Manual sanity check (Day 3, later)

If output is broken (garbage, repetition, parroting the prompt back), the cause
is almost always a chat-template mismatch between training and inference, not a
bad learning rate. Check the template before touching hyperparameters.

In [ ]:
HAND_PICKED_PROMPTS = [
    "Explain what a hash map is to someone who has never programmed before.",
    "Write a short poem about the ocean.",
    "What are three tips for staying focused while studying?",
    "Translate 'good morning' into French, Spanish, and German.",
    "Summarize the plot of Romeo and Juliet in two sentences.",
    "List the first five prime numbers.",
    "Give me a recipe idea using chicken, rice, and broccoli.",
    "What is the capital of Australia?",
    "Write a one-line joke about programmers.",
    "Explain the difference between a list and a tuple in Python.",
]

_sanity_model = load_with_adapter(SFT_ADAPTER_DIR)
_sanity_results = generate_batch(_sanity_model, HAND_PICKED_PROMPTS)
free_model(_sanity_model)

for r in _sanity_results:
    print("=" * 80)
    print("PROMPT:  ", r["prompt"])
    print("RESPONSE:", r["response"])
    print("hit_eos:", r["hit_eos"])

n_eos = sum(r["hit_eos"] for r in _sanity_results)
print(f"\n{n_eos}/{len(_sanity_results)} generations stopped on EOS.")


## 6. Base vs. SFT eval (Day 4 — run this now, not later)

Base → SFT should show a large, obvious win rate. If it doesn't, something
upstream is wrong and you want to know that with five days of runway left.

In [ ]:
_base_model = load_base_model()
base_gens = generate_batch(_base_model, eval_prompts)
free_model(_base_model)

_sft_model = load_with_adapter(SFT_ADAPTER_DIR)
sft_gens = generate_batch(_sft_model, eval_prompts)
free_model(_sft_model)

base_vs_sft_raw = [
    {**judge_pair(p, a["response"], b["response"], backend=JUDGE_BACKEND), "prompt": p}
    for p, a, b in zip(eval_prompts, base_gens, sft_gens)
]
base_vs_sft_summary = summarize_results(base_vs_sft_raw, name_a="base", name_b="sft")
print(json.dumps(base_vs_sft_summary, indent=2))

with open(f"{EVAL_DIR}/base_vs_sft.json", "w") as f:
    json.dump({"summary": base_vs_sft_summary, "raw": base_vs_sft_raw}, f, indent=2)


## 7. DPO (Day 5)

With LoRA, `ref_model=None` makes TRL compute reference logits by temporarily
disabling the adapters on the same frozen base weights — no second copy of the
model in memory. This is why DPO fits on a 16GB T4 without QLoRA.

In [ ]:
from trl import DPOConfig, DPOTrainer

_dpo_base = load_base_model()
dpo_policy = PeftModel.from_pretrained(_dpo_base, SFT_ADAPTER_DIR, is_trainable=True)
dpo_policy.config.use_cache = False

dpo_config = DPOConfig(
    output_dir=DPO_ADAPTER_DIR,
    beta=DPO_BETA,
    learning_rate=DPO_LEARNING_RATE,
    per_device_train_batch_size=DPO_TRAIN_BATCH_SIZE,
    gradient_accumulation_steps=DPO_GRAD_ACCUM_STEPS,
    num_train_epochs=DPO_EPOCHS,
    fp16=True,
    max_length=DPO_MAX_LENGTH,
    max_prompt_length=DPO_MAX_PROMPT_LENGTH,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    logging_steps=10,
    save_strategy="epoch",
    report_to="none",
)

dpo_trainer = DPOTrainer(
    model=dpo_policy,
    ref_model=None,  # LoRA: reference logits computed with adapters disabled
    args=dpo_config,
    train_dataset=dpo_train_dataset,
    processing_class=tokenizer,
)

dpo_trainer.train()
# Watch rewards/accuracies (should climb above 0.5) and rewards/margins
# (should widen) in the logs above. A flatline means DPO isn't learning anything.


In [ ]:
dpo_trainer.model.save_pretrained(DPO_ADAPTER_DIR)
tokenizer.save_pretrained(DPO_ADAPTER_DIR)
print(f"DPO adapter saved to {DPO_ADAPTER_DIR}")

if hf_token:
    dpo_trainer.model.push_to_hub(DPO_ADAPTER_REPO)
    tokenizer.push_to_hub(DPO_ADAPTER_REPO)
    print(f"Pushed DPO adapter to https://huggingface.co/{DPO_ADAPTER_REPO}")
else:
    print("Skipping Hub push (no HF_TOKEN). Adapter is saved locally.")


## 8. Full three-way evaluation + objective metrics (Day 6)

Base vs. SFT vs. SFT+DPO, judged pairwise three ways, plus the §6 objective
metrics: reward accuracy on the preference holdout, mean response length, and
format adherence.

In [ ]:
import torch.nn.functional as F


def _sequence_logprob(model, prompt, completion, device):
    full_text = prompt + completion
    prompt_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)
    full_ids = tokenizer(full_text, return_tensors="pt").input_ids.to(device)

    with torch.no_grad():
        logits = model(full_ids).logits

    completion_start = prompt_ids.shape[1]
    shift_logits = logits[:, completion_start - 1 : -1, :]
    shift_labels = full_ids[:, completion_start:]

    log_probs = F.log_softmax(shift_logits.float(), dim=-1)
    token_log_probs = log_probs.gather(-1, shift_labels.unsqueeze(-1)).squeeze(-1)
    return token_log_probs.sum().item()


def reward_accuracy(model, pref_holdout, beta=DPO_BETA):
    device = next(model.parameters()).device
    correct = 0
    for pair in pref_holdout:
        prompt, chosen, rejected = pair["prompt"], pair["chosen"], pair["rejected"]
        policy_chosen = _sequence_logprob(model, prompt, chosen, device)
        policy_rejected = _sequence_logprob(model, prompt, rejected, device)
        with model.disable_adapter():
            ref_chosen = _sequence_logprob(model, prompt, chosen, device)
            ref_rejected = _sequence_logprob(model, prompt, rejected, device)
        reward_chosen = beta * (policy_chosen - ref_chosen)
        reward_rejected = beta * (policy_rejected - ref_rejected)
        if reward_chosen > reward_rejected:
            correct += 1
    return correct / len(pref_holdout)


def mean_response_length(generations):
    lengths = [len(g["response"].split()) for g in generations]
    return sum(lengths) / len(lengths) if lengths else float("nan")


def format_adherence_rate(generations):
    hits = [g["hit_eos"] for g in generations]
    return sum(hits) / len(hits) if hits else float("nan")


In [ ]:
# Re-generate from base and SFT for a clean three-way comparison (fresh sampling)
_base_model = load_base_model()
base_gens = generate_batch(_base_model, eval_prompts)
free_model(_base_model)

_sft_model = load_with_adapter(SFT_ADAPTER_DIR)
sft_gens = generate_batch(_sft_model, eval_prompts)
free_model(_sft_model)

dpo_model = load_with_adapter(DPO_ADAPTER_DIR)
dpo_gens = generate_batch(dpo_model, eval_prompts)

r_acc = reward_accuracy(dpo_model, pref_holdout)
free_model(dpo_model)

print(f"Reward accuracy on preference holdout: {r_acc:.3f}")


In [ ]:
def run_pairing(gens_a, gens_b, name_a, name_b):
    raw = [
        {**judge_pair(p, a["response"], b["response"], backend=JUDGE_BACKEND), "prompt": p}
        for p, a, b in zip(eval_prompts, gens_a, gens_b)
    ]
    return summarize_results(raw, name_a=name_a, name_b=name_b), raw


base_vs_sft_summary, base_vs_sft_raw = run_pairing(base_gens, sft_gens, "base", "sft")
sft_vs_dpo_summary, sft_vs_dpo_raw = run_pairing(sft_gens, dpo_gens, "sft", "dpo")
base_vs_dpo_summary, base_vs_dpo_raw = run_pairing(base_gens, dpo_gens, "base", "dpo")

objective_metrics = {
    "reward_accuracy_pref_holdout": r_acc,
    "mean_response_length": {
        "base": mean_response_length(base_gens),
        "sft": mean_response_length(sft_gens),
        "dpo": mean_response_length(dpo_gens),
    },
    "format_adherence_rate": {
        "base": format_adherence_rate(base_gens),
        "sft": format_adherence_rate(sft_gens),
        "dpo": format_adherence_rate(dpo_gens),
    },
}

report = {
    "win_rates": {
        "base_vs_sft": base_vs_sft_summary,
        "sft_vs_dpo": sft_vs_dpo_summary,
        "base_vs_dpo": base_vs_dpo_summary,
    },
    "objective_metrics": objective_metrics,
}

print(json.dumps(report, indent=2))

with open(f"{EVAL_DIR}/three_way_summary.json", "w") as f:
    json.dump(report, f, indent=2)
with open(f"{EVAL_DIR}/three_way_raw_judgments.json", "w") as f:
    json.dump(
        {"base_vs_sft": base_vs_sft_raw, "sft_vs_dpo": sft_vs_dpo_raw, "base_vs_dpo": base_vs_dpo_raw},
        f, indent=2,
    )
print(f"Results written to {EVAL_DIR}/three_way_summary.json")


In [ ]:
# A few side-by-side examples for the README write-up.
for i in range(3):
    print("=" * 100)
    print("PROMPT:", eval_prompts[i])
    print("-- base --\n", base_gens[i]["response"])
    print("-- SFT --\n", sft_gens[i]["response"])
    print("-- SFT+DPO --\n", dpo_gens[i]["response"])


## 9. Write-up (Day 7)

Copy the printed `report` JSON and the side-by-side examples above into the
project README's "Results" section. Also note, in a sentence or two:

- What did DPO actually change qualitatively — tone, verbosity, refusal behavior?
- Did the reward-accuracy metric clear 50%? Did mean response length move in the
  direction the preference axis predicted?
- Where does this evaluation remain vulnerable to bias, even with randomized order?

Push the LoRA adapters (already pushed to the Hub above if `HF_TOKEN` was set),
this notebook, and the eval JSON files under `/kaggle/working/eval/` to your
public GitHub repo alongside the training scripts.
